# 📓 02 — Policy AbstractionThe policy layer is the heart of strands-robots. Every VLA provider (GR00T, ACT, SmolVLA, custom)implements the same `Policy` ABC, so robot control code never changes when you swap models.

## The Policy ABC```pythonclass Policy(ABC):    async def get_actions(self, observation_dict, instruction, **kwargs) -> list[dict[str, Any]]:        ...    def get_actions_sync(self, observation_dict, instruction, **kwargs) -> list[dict[str, Any]]:        ...  # convenience wrapper    def set_robot_state_keys(self, robot_state_keys: list[str]) -> None:        ...    @property    def provider_name(self) -> str:        ...```Key design decisions:- **Async-first**: `get_actions()` is async — works with real robot event loops- **Sync wrapper**: `get_actions_sync()` handles event loop detection automatically- **Observation as dict**: `{camera_name: ndarray, joint_name: float}` — universal format- **Action as list of dicts**: Each dict = one timestep, keys = actuator names

In [ ]:
from strands_robots import Policy, MockPolicy, create_policy# MockPolicy generates smooth sinusoidal trajectories for testingmock = MockPolicy()print(f"Provider: {mock.provider_name}")# Set robot state keys (tells policy what joints exist)mock.set_robot_state_keys(["shoulder", "elbow", "wrist", "gripper"])print(f"State keys: {mock.robot_state_keys}")

In [ ]:
import asyncio# get_actions returns a list of action dicts (one per timestep in the chunk)observation = {"shoulder": 0.0, "elbow": 0.5, "wrist": -0.3, "gripper": 0.0}actions = await mock.get_actions(observation, "pick up the cube")print(f"Action chunk size: {len(actions)}")print(f"First action: { {k: round(v, 4) for k, v in actions[0].items()} }")print(f"Keys match state keys: {list(actions[0].keys()) == mock.robot_state_keys}")

In [ ]:
# Sync wrapper — auto-detects running event loopactions_sync = mock.get_actions_sync(observation, "pick up the cube")print(f"Sync actions: {len(actions_sync)} steps")print(f"Values are smooth sinusoids: {[round(actions_sync[i]['shoulder'], 3) for i in range(8)]}")

## create_policy() FactoryThe factory resolves policy providers from the JSON registry. Supports:- Provider names: `"mock"`, `"groot"`, `"lerobot_local"`- Smart strings: `"lerobot/act_aloha_sim"` → auto-detects provider- ZMQ URLs: `"zmq://localhost:5555"` → GR00T service mode

In [ ]:
from strands_robots import create_policy# Create by provider namemock_policy = create_policy("mock")print(f"Mock: {mock_policy.provider_name}")# List all available providersfrom strands_robots.policies import list_providersprint(f"\nAvailable providers: {list_providers()}")

## Writing a Custom PolicyImplement the Policy ABC to create your own VLA provider.

In [ ]:
import numpy as npfrom strands_robots.policies import Policy, register_policyclass RandomPolicy(Policy):    """Random actions for testing — simpler than MockPolicy."""    def __init__(self, action_scale: float = 0.1, **kwargs):        self.action_scale = action_scale        self.state_keys: list[str] = []    @property    def provider_name(self) -> str:        return "random"    def set_robot_state_keys(self, keys: list[str]) -> None:        self.state_keys = keys    async def get_actions(self, observation_dict, instruction, **kwargs):        actions = []        for _ in range(8):  # 8-step action chunk            action = {key: float(np.random.uniform(-self.action_scale, self.action_scale))                      for key in self.state_keys}            actions.append(action)        return actions# Register it so create_policy("random") worksregister_policy("random", lambda: RandomPolicy, aliases=["rnd"])# Now use itrnd = create_policy("random", action_scale=0.05)rnd.set_robot_state_keys(["j1", "j2", "j3"])actions = rnd.get_actions_sync({}, "test")print(f"Random actions: {len(actions)} steps")print(f"Sample: { {k: round(v, 4) for k, v in actions[0].items()} }")